In [23]:
import requests
import pandas as pd
import json

In [24]:
# Define the time period for the weather data
start_date = "2024-10-01"
end_date = "2026-09-30"

In [25]:
# Define the locations and their geographic coordinates
locations = {
    "Grenoble": {
        "latitude": 45.18,
        "longitude": 5.72
    },
    "Chamrousse": {
        "latitude": 45.13,
        "longitude": 5.88
    },
    "Voiron": {
        "latitude": 45.36,
        "longitude": 5.59
    }
}

In [26]:
# Define the Open-Meteo Historical Weather API endpoint
url = "https://archive-api.open-meteo.com/v1/archive"

In [27]:
# Define the hourly weather variables to retrieve
weather_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "snow_depth"
]

In [28]:
# Define the parameters for a first API request for Grenoble
params = {
    "latitude": locations["Grenoble"]["latitude"],
    "longitude": locations["Grenoble"]["longitude"],
    "start_date": start_date,
    "end_date": end_date,
    "hourly": weather_variables
}

In [29]:
# Send a GET request to the Open-Meteo API
response = requests.get(url, params=params)

# Display the HTTP status code
print(response.status_code)

200


In [30]:
# Convert the API response to JSON
data = response.json()

# Display the main keys of the JSON response
print(data.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])


In [31]:
# Inspect the hourly weather data returned by the API
print(data["hourly"].keys())

dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'snow_depth'])


In [32]:
# Display the first five timestamps and temperatures
print(data["hourly"]["time"][:5])
print(data["hourly"]["temperature_2m"][:5])

['2024-10-01T00:00', '2024-10-01T01:00', '2024-10-01T02:00', '2024-10-01T03:00', '2024-10-01T04:00']
[16.1, 15.8, 15.8, 15.7, 15.7]


In [33]:
# Convert the hourly JSON data into a Pandas DataFrame
df = pd.DataFrame(data["hourly"])

# Display the first five rows
df.head()

,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,snow_depth
0,2024-10-01T00:00,16.1,82,0.0,5.4,0.0
1,2024-10-01T01:00,15.8,85,0.0,4.9,0.0
2,2024-10-01T02:00,15.8,84,0.0,4.7,0.0
3,2024-10-01T03:00,15.7,86,0.0,5.0,0.0
4,2024-10-01T04:00,15.7,86,0.0,4.9,0.0


In [34]:
# Display the number of rows and columns in the DataFrame
print("Shape:", df.shape)

Shape: (17520, 6)


In [35]:
# Check the first and last timestamps in the dataset
print("First timestamp:", df["time"].min())
print("Last timestamp:", df["time"].max())

First timestamp: 2024-10-01T00:00
Last timestamp: 2026-09-30T23:00


In [36]:
# Create an empty list to store the data from all locations
all_data = []

# Loop through each location
for name, coordinates in locations.items():

    # Define the parameters of the API request for the current location
    params = {
        "latitude": coordinates["latitude"],
        "longitude": coordinates["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m",
            "snow_depth"
        ]
    }

    # Send the request to the Open-Meteo API
    response = requests.get(url, params=params)

    # Convert the API response to JSON
    data = response.json()

    # Convert the hourly weather data into a Pandas DataFrame
    df_location = pd.DataFrame(data["hourly"])

    # Add the location name to the DataFrame
    df_location["location"] = name

    # Add this DataFrame to the list
    all_data.append(df_location)

    # Display the location and the shape of its DataFrame
    print(name, df_location.shape)

Grenoble (17520, 7)
Chamrousse (17520, 7)
Voiron (17520, 7)


In [37]:
# Combine the data from all locations into one DataFrame
df_all = pd.concat(all_data, ignore_index=True)

# Display the shape and the first rows of the final DataFrame
print("Shape:", df_all.shape)
df_all.head()

Shape: (52560, 7)


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,snow_depth,location
0,2024-10-01T00:00,16.1,82,0.0,5.4,0.0,Grenoble
1,2024-10-01T01:00,15.8,85,0.0,4.9,0.0,Grenoble
2,2024-10-01T02:00,15.8,84,0.0,4.7,0.0,Grenoble
3,2024-10-01T03:00,15.7,86,0.0,5.0,0.0,Grenoble
4,2024-10-01T04:00,15.7,86,0.0,4.9,0.0,Grenoble


In [38]:
# Save the final DataFrame as a Parquet file
df_all.to_parquet("weather_data.parquet", index=False)

# Display the shape of the saved dataset
print("Shape:", df_all.shape)

Shape: (52560, 7)


In [39]:
# Create a log describing the data acquisition
acquisition_log = {
    "source": "Open-Meteo",
    "endpoint": url,

    # Document the scope revision made for M1
    "scope_revision": (
        "The registered scope referred to Grenoble and nearby weather stations. "
        "For M1, the scope is revised to Grenoble and nearby geographic locations "
        "queried by latitude and longitude through the Open-Meteo Historical Weather API."
    ),

    "start_date": start_date,
    "end_date": end_date,
    "locations": locations,
    "variables": weather_variables,
    "parquet_file": "weather_data.parquet",
    "shape": list(df_all.shape)
}


In [40]:
# Save the acquisition log as a JSON file
with open("acquisition_log.json", "w") as file:
    json.dump(acquisition_log, file, indent=4)

print("Acquisition log saved.")

Acquisition log saved.


In [41]:
# Read the acquisition log to check its content
with open("acquisition_log.json", "r") as file:
    log = json.load(file)

print(log)

{'source': 'Open-Meteo', 'endpoint': 'https://archive-api.open-meteo.com/v1/archive', 'scope_revision': 'The registered scope referred to Grenoble and nearby weather stations. For M1, the scope is revised to Grenoble and nearby geographic locations queried by latitude and longitude through the Open-Meteo Historical Weather API.', 'start_date': '2024-10-01', 'end_date': '2026-09-30', 'locations': {'Grenoble': {'latitude': 45.18, 'longitude': 5.72}, 'Chamrousse': {'latitude': 45.13, 'longitude': 5.88}, 'Voiron': {'latitude': 45.36, 'longitude': 5.59}}, 'variables': ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'snow_depth'], 'parquet_file': 'weather_data.parquet', 'shape': [52560, 7]}
